# Ejercicios UD1 · Bloque 3 — Calidad de datos

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos: estructura, variables y calidad
**Alumno/a:**
**Fecha de entrega:**

---

Este cuaderno recoge los ejercicios del **Bloque 3** de la UD1. Están organizados por sesiones, igual que los cuadernos de teoría:

| Sesión | Cuaderno de teoría | Ejercicios |
|---|---|---|
| 1 | `B3_S1_Valores_Ausentes_y_Outliers` | 1.1 – 1.5 |
| 2 | `B3_S2_Datos_Incorrectos_y_Cobertura` | 2.1 – 2.6 |

Seguimos con los pacientes de **enfermedad cardiaca** del Bloque 2, donde dejamos pendiente una tarea: en el ejercicio 3.4 apuntaste los problemas de calidad «que hay que tratar en el Bloque 3». Ahora toca diagnosticarlos a fondo.

- En la **Sesión 1** trabajas con `heart.csv`, el mismo fichero del Bloque 2: valores ausentes disfrazados, mecanismos de ausencia y valores atípicos.
- En la **Sesión 2** trabajas con `heart_sucio.csv`: los **mismos pacientes**, pero tal como llegarían si tres centros sanitarios exportaran sus registros cada uno a su manera. Tendrás que cargarlo, encontrar los errores, corregirlos y dejar un dataset limpio y documentado.

Algunos ejercicios son preguntas de reflexión (responde en la celda de texto) y otros son de código (escribe tu solución donde pone `TODO`). Los ejercicios se encadenan: los DataFrames que preparas en un ejercicio se usan en los siguientes.

> **Antes de empezar:** copia este cuaderno a tu carpeta `mis_cuadernos/` y trabaja sobre la copia, así `git pull` no te dará conflictos. Copia también la carpeta `datos/` (o ajusta la ruta `DATOS` de la celda de carga).
>
> **Cómo entregar:** reinicia el kernel y ejecuta todas las celdas (**Restart** → **Run All**), comprueba que no hay errores, guarda y sube el fichero `.ipynb` al aula virtual. Sube también los dos CSV que genera el ejercicio 2.6.

## Criterios de evaluación

Esta relación de ejercicios evalúa los siguientes criterios de evaluación del módulo (Real Decreto 183/2026, redacción literal):

> **RA1.** Identifica la estructura y variables en los datos en el dominio de aplicación, así como su cobertura y calidad del conjunto, aplicando técnicas exploratorias mediante análisis exploratorios de datos (AED) para la selección de atributos clave en el entrenamiento posterior.

| CE | Criterio de evaluación | Ejercicios |
|:--:|------------------------|:----------:|
| **1.b** | Se han detectado datos incorrectamente introducidos, ausencia de valores, una mala codificación de las variables o valores atípicos. | Todos |
| **1.c** | Se han identificado la estructura, variables y relaciones de los conjuntos de datos aplicando técnicas exploratorias. | 1.2, 1.3, 1.5, 2.5 |
| **1.d** | Se han identificado entidades, volumetrías, relaciones y atributos, para documentar cada conjunto de datos, describiendo sus características. | 2.1, 2.2, 2.5, 2.6 |

In [ ]:
# Librerías que se usan en todo el cuaderno
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from pandas.api.types import CategoricalDtype

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías cargadas')

## Los datos

`heart.csv` es el dataset **Heart Failure Prediction** que ya conoces del Bloque 2: 918 pacientes de cinco hospitales estudiados por sospecha de enfermedad cardiaca, con 11 variables clínicas y la variable objetivo `HeartDisease`. Si necesitas repasar el significado de las variables, vuelve a la tabla del cuaderno de ejercicios del Bloque 2.

La celda siguiente lo carga (y lo descarga si no lo tienes). Como en el Bloque 2:

> `heart_raw` es la copia **original** y no se toca en todo el cuaderno. Trabaja siempre sobre copias.

In [ ]:
DATOS = Path('datos')
URL_HEART = 'https://raw.githubusercontent.com/imteekay/heart-failure-prediction/main/heart.csv'
ruta_heart = DATOS / 'heart.csv'

if not ruta_heart.exists():
    print('Descargando heart.csv...')
    DATOS.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL_HEART, ruta_heart)

heart_raw = pd.read_csv(ruta_heart)   # copia original: no la modifiques
heart = heart_raw.copy()              # copia de trabajo

NUMERICAS = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
print('✅ Dataset cargado:', heart.shape)

---
# Sesión 1 — Valores ausentes y valores atípicos

## Ejercicio 1.1 — Los ausentes que no se ven

En el Bloque 2 descubriste que `info()` decía «0 nulos» y, sin embargo, había pacientes con colesterol 0 mg/dL. Vamos a hacer el diagnóstico de ausentes completo.

**Parte A — Recuento.**

1. Cuenta los `NaN` por columna de `heart` con `isnull()`.
2. Decide en qué columnas numéricas un **0 es físicamente imposible** y guárdalas en una lista `COLS_CERO_IMPOSIBLE`. Revisa las cinco de `NUMERICAS` y también `FastingBS`.
3. Crea `heart_na`, una copia de `heart` en la que los ceros de esas columnas se sustituyan por `NaN`.
4. Construye una tabla con el número y el porcentaje de ausentes de las columnas de `heart_na` que tengan alguno, ordenada de mayor a menor.
5. Cuenta los ausentes **por fila**: ¿cuántas filas tienen 0, 1, 2… ausentes?

In [ ]:
# TODO: NaN por columna en heart

# TODO: COLS_CERO_IMPOSIBLE = [...]

# TODO: heart_na con los ceros imposibles como NaN

# TODO: tabla de ausentes (n y %) de heart_na

# TODO: ausentes por fila

**Parte B — Visualización con `missingno`.**
Dibuja, cada uno en su celda, `msno.matrix`, `msno.bar` y `msno.heatmap` de `heart_na`. Pon título a cada gráfico.

In [ ]:
# TODO: msno.matrix

In [ ]:
# TODO: msno.bar

In [ ]:
# TODO: msno.heatmap

**Preguntas:**

1. ¿Por qué `isnull()` no detecta estos ausentes? Además del 0, ¿qué otras formas de «disfrazar» un ausente conoces? ¿Qué parámetro de `pd.read_csv()` permite indicarlas al cargar el fichero?
2. ¿Por qué **no** has incluido `Oldpeak` ni `FastingBS` en `COLS_CERO_IMPOSIBLE`?
3. Hay una fila con dos ausentes. ¿Qué tiene de especial? ¿Es casualidad?
4. ¿Qué te dice `msno.matrix` sobre **dónde** están los ausentes? ¿Te parece útil el `msno.heatmap` con estos datos? ¿Por qué?

*Escribe aquí tus respuestas:*

## Ejercicio 1.2 — ¿Por qué falta el colesterol? Diagnóstico del mecanismo

Antes de decidir qué hacer con un ausente hay que averiguar **por qué falta** (MCAR, MAR o MNAR). La estrategia es la del Ejemplo 3 de la teoría: dividir a los pacientes en dos grupos, *con* y *sin* colesterol, y comparar el resto de variables.

**Parte A — Comparar los dos grupos.**
Añade a `heart_na` una columna booleana `chol_ausente`. Después:

1. Calcula el número de pacientes y el **porcentaje de enfermos** en cada grupo.
2. Calcula la media de `Age`, `RestingBP`, `MaxHR` y `Oldpeak` en cada grupo.
3. Con `pd.crosstab(..., normalize='index')`, compara la distribución (en %) de `Sex`, `ChestPainType`, `ExerciseAngina` y `ST_Slope` en los dos grupos.

In [ ]:
# TODO: columna chol_ausente

# TODO: n y % de enfermos por grupo

# TODO: medias de las numéricas por grupo

# TODO: crosstabs normalizados por fila

**Parte B — ¿Dónde están los ausentes?**
Recuerda que las filas del fichero están ordenadas por hospital de origen. Divide las filas en **10 tramos consecutivos** del mismo tamaño (pista: `pd.cut(heart_na.index, bins=10, labels=False)`) y dibuja en una figura de 1 × 2:

- a la izquierda, el **porcentaje de colesterol ausente** en cada tramo (barras);
- a la derecha, un boxplot de `MaxHR` según `chol_ausente`.

In [ ]:
# TODO: % de ausentes por tramo de filas y boxplot de MaxHR

**Parte C — ¿Qué pasa si eliminamos las filas?**
La solución «rápida» sería `dropna()`. Construye una tabla que compare `heart_na` con `heart_na.dropna(subset=['Cholesterol'])` en: número de pacientes, % de enfermos, % de mujeres y % de pacientes asintomáticos (`ChestPainType == 'ASY'`).

In [ ]:
# TODO: tabla comparativa antes / después de dropna

**Preguntas:**

1. ¿Son los ausentes de `Cholesterol` **MCAR**? Justifícalo con al menos dos cifras de la Parte A.
2. Según la Parte B, ¿de qué depende realmente que falte el colesterol? ¿Lo clasificarías como MAR o como MNAR? Ten en cuenta que la variable de la que depende **no está** en el dataset.
3. ¿Qué le pasa al dataset si eliminas esas filas? ¿Qué pacientes salen perjudicados? ¿Qué consecuencias tendría para un modelo entrenado después?
4. Sin hacerlo todavía (el tratamiento es de la UD3), ¿qué alternativas a eliminar las filas se te ocurren?

*Escribe aquí tus respuestas:*

## Ejercicio 1.3 — Laboratorio: los tres mecanismos de ausencia

En los datos reales nunca sabemos el valor que falta. Aquí haremos trampa: partimos de los pacientes **con** colesterol, así conocemos la media verdadera, y **provocamos** ausentes con cada mecanismo para ver qué efecto tiene.

**Parte A — Simulador.**
Crea `completo = heart[heart['Cholesterol'] > 0].copy()` y escribe una función `simular_ausentes(df, col, mecanismo, pct=0.25, seed=42)` que devuelva una **copia** de `df` con el `pct` de los valores de `col` cambiados a `NaN`. Las filas se eligen con `df.sample(n=k, weights=pesos, random_state=seed)`, con estos pesos:

| Mecanismo | Pesos | Qué simula |
|---|---|---|
| `'MCAR'` | todos 1 | Se pierden tubos de sangre al azar |
| `'MAR'` | 5 si `Age >= 60`, 1 si no | A los mayores se les hace menos la analítica |
| `'MNAR'` | 5 si `Cholesterol >= 260`, 1 si no | Los resultados altos se mandan a repetir y no llegan |

In [ ]:
# TODO: completo y función simular_ausentes
# def simular_ausentes(df, col, mecanismo, pct=0.25, seed=42):
#     ...

**Parte B — ¿Qué mecanismo sesga la media?**
Para cada mecanismo, construye una fila de una tabla con:

- `media_observada`: media de `Cholesterol` con los valores que quedan;
- `sesgo`: `media_observada` − media verdadera (la de `completo`);
- `edad_ausentes` y `edad_presentes`: media de `Age` en las filas con y sin ausente (la herramienta de diagnóstico del ejercicio 1.2).

Añade una primera fila `'verdadero'` con los valores de `completo`.

In [ ]:
# TODO: tabla comparativa de los tres mecanismos

**Parte C — Distribuciones.**
Dibuja en un mismo gráfico la curva KDE de `Cholesterol` de `completo` y la de los valores observados con cada mecanismo. Usa una línea más gruesa para la distribución verdadera.

In [ ]:
# TODO: KDE de la distribución verdadera y de las tres simuladas

**Preguntas:**

1. ¿Qué mecanismo sesga claramente la media? ¿En qué sentido y por qué? En el caso MAR la media apenas cambia: ¿por qué? (Pista: calcula la correlación entre `Age` y `Cholesterol` en `completo`.)
2. En el caso MAR, ¿podrías haber descubierto el mecanismo con los datos observados? ¿Cómo?
3. En el caso MNAR, compara `edad_ausentes` y `edad_presentes`. ¿Te sirve esa comparación para detectarlo? ¿Por qué MNAR es el caso más peligroso?
4. Repite la Parte B con `seed=1`, `seed=2` y `seed=3`. ¿El sesgo de MCAR es exactamente 0? ¿Qué diferencia hay entre el «sesgo» de MCAR y el de MNAR?

*Escribe aquí tus respuestas:*

## Ejercicio 1.4 — Tres detectores de atípicos

**Parte A — Funciones.**
Escribe tres funciones que reciban una `Series` y devuelvan una **máscara booleana** (misma longitud e índice que la serie; los `NaN` cuentan como `False`):

- `atipicos_iqr(serie, factor=1.5)`: fuera de `[Q1 − factor·IQR, Q3 + factor·IQR]`.
- `atipicos_z(serie, umbral=3)`: `|z| > umbral`, con `z = (x − media) / desviación típica`.
- `atipicos_zmod(serie, umbral=3.5)`: z-score modificado, `0,6745 · |x − mediana| / MAD > umbral`.

> Devolver una máscara (y no un número) permite después contar, filtrar filas o combinar métodos con `&` y `|`.

In [ ]:
# TODO: atipicos_iqr, atipicos_z, atipicos_zmod

**Parte B — Comparación.**
Aplica los tres métodos a las variables de `NUMERICAS` de `heart_na` y muestra una tabla con una fila por variable, el número de atípicos de cada método y la asimetría (`skew()`).

In [ ]:
# TODO: tabla de atípicos por método

**Parte C — Los límites, a la vista.**
Dibuja el histograma de `Cholesterol` de `heart_na` y marca con líneas verticales de un color distinto los límites de cada método (calcula los límites a mano a partir de las fórmulas).

In [ ]:
# TODO: histograma de Cholesterol con los límites de los tres métodos

**Parte D — El efecto de los ceros.**
Repite el recuento de atípicos de `Cholesterol` con los tres métodos, pero sobre `heart` (con los ceros). Muestra los dos resultados juntos y, para cada método, cuántos de los atípicos detectados son ceros.

In [ ]:
# TODO: atípicos de Cholesterol con y sin ceros

**Preguntas:**

1. ¿Qué método detecta más atípicos en `RestingBP` y en `Cholesterol`? ¿Y menos? Relaciónalo con la asimetría.
2. El z-score supone que los datos siguen una distribución aproximadamente normal. ¿En qué variables de `NUMERICAS` es razonable usarlo?
3. En la Parte D, ¿qué método «ve» los 172 ceros como atípicos y cuál no? Explica por qué el z-score se deja engañar (pista: ¿qué les pasa a la media y a la desviación típica cuando hay 172 ceros?).
4. Mirando la Parte C, ¿por qué el z-score modificado y el IQR dan límites parecidos y el z-score no?

*Escribe aquí tus respuestas:*

## Ejercicio 1.5 — ¿Error o valor real?

Un atípico **estadístico** no es necesariamente un **error**. Para separarlos necesitamos conocimiento del dominio. Estos son unos rangos de plausibilidad orientativos para adultos (no son criterios médicos):

In [ ]:
# Rangos de plausibilidad orientativos: (mínimo, máximo)
REGLAS = {
    'Age':         (18, 100),    # años
    'RestingBP':   (60, 250),    # mm Hg
    'Cholesterol': (80, 650),    # mg/dL
    'MaxHR':       (50, 220),    # ppm
    'Oldpeak':     (-3, 7),      # mm
}

**Parte A — Reglas de dominio.**
Escribe una función `fuera_de_dominio(df, reglas)` que devuelva una tabla con una fila por variable y las columnas `minimo`, `maximo`, `n_por_debajo` y `n_por_encima`. Aplícala a `heart` y a `heart_na`.

In [ ]:
# TODO: función fuera_de_dominio y aplicación a heart y heart_na

**Parte B — Los casos extremos.**
Para cada variable de `NUMERICAS`, muestra los atípicos IQR de `heart_na` que **sí** están dentro del rango plausible: cuántos son y sus valores mínimo y máximo. Después muestra la fila completa del paciente con el colesterol más alto y la del paciente con la `MaxHR` más baja.

In [ ]:
# TODO: atípicos IQR dentro del dominio

# TODO: filas de los pacientes más extremos

**Parte C — Atípicos multivariantes.**
Un valor puede ser normal en cada variable por separado y raro en **combinación**. Una regla clásica estima la frecuencia cardiaca máxima teórica de una persona como **220 − edad**. Calcula `pct_fc_teorica = MaxHR / (220 − Age) * 100` (qué porcentaje de su máximo teórico alcanzó cada paciente en la prueba) y:

1. Cuenta cuántos pacientes **superan el 100 %** de su máximo teórico y cuántos de ellos son atípicos IQR en `Age` o en `MaxHR`.
2. Cuenta cuántos **no llegan al 85 %** (en cardiología, no alcanzar el 85 % se llama *incompetencia cronotrópica*) y compara su porcentaje de enfermos con el del resto.
3. Dibuja `MaxHR` frente a `Age` coloreando en tres grupos (`< 85 %`, `85-100 %`, `> 100 %`) y añade la recta `220 − edad` como referencia.

In [ ]:
# TODO: pct_fc_teorica, recuentos y gráfico

**Parte D — Informe de calidad.**
Escribe una función `informe_calidad(df, reglas)` que devuelva un DataFrame con una fila por columna **numérica** y las columnas `dtype`, `n_ausentes`, `pct_ausentes`, `n_atipicos_iqr`, `pct_atipicos_iqr` y `n_fuera_dominio` (esta última solo para las columnas que estén en `reglas`; en las demás, 0). Aplícala a `heart_raw` y a `heart_na` y muestra los dos resultados.

> La usarás otra vez en el ejercicio 2.6.

In [ ]:
# TODO: función informe_calidad

**Preguntas:**

1. Una vez convertidos los ceros en `NaN`, ¿queda algún valor fuera del rango plausible? ¿Qué conclusión sacas sobre los atípicos de este dataset?
2. Busca qué significa un `Oldpeak` negativo y uno de 6 mm. ¿Los eliminarías?
3. Los pacientes que superan el 100 % de su frecuencia máxima teórica, ¿son errores? ¿Por qué no aparecen como atípicos si miras la edad y la frecuencia por separado? (Pista: busca qué precisión tiene la fórmula 220 − edad.)
4. ¿Qué te dice la Parte C.2 sobre la utilidad de crear variables **derivadas** durante el AED?
5. Completa la tabla con al menos cuatro casos que hayas encontrado en este ejercicio y en el 1.4:

| Variable | Valor(es) | ¿Atípico estadístico? | ¿Fuera de dominio? | Decisión: error / extremo real / dudoso | Justificación |
|---|---|---|---|---|---|
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |

*Escribe aquí tus respuestas:*

---
# Sesión 2 — Datos incorrectos, codificación y cobertura

## El fichero `heart_sucio.csv`

Imagina que los datos no te llegan ya preparados de Kaggle, sino que **tres centros** (*Hospital Norte*, *Clínica Sur* y *Hospital Este*) te envían una exportación de sus registros. Cada centro usa su propio programa, su propio Excel y sus propias costumbres. El resultado es `datos/heart_sucio.csv`, que viene con el repositorio.

> Es un fichero **simulado para practicar**: contiene los mismos 918 pacientes de `heart.csv`, pero con los errores típicos de una exportación real introducidos a propósito. Los nombres de los centros son inventados.

Además de las 12 variables que ya conoces, tiene cuatro columnas nuevas:

| Variable | Descripción |
|---|---|
| `id_paciente` | Identificador del paciente (`P0001`, `P0002`…) |
| `centro` | Centro que envió el registro |
| `fecha_visita` | Fecha en la que se hicieron las pruebas |
| `observaciones` | Comentario libre del personal sanitario |

El fichero se exportó el **30 de junio de 2026**.

In [ ]:
RUTA_SUCIO = DATOS / 'heart_sucio.csv'
FECHA_EXPORTACION = pd.Timestamp('2026-06-30')
print('¿Existe el fichero?', RUTA_SUCIO.exists())

## Ejercicio 2.1 — Un fichero que no quiere cargarse

**Parte A — Encoding y separador.**

1. Intenta cargar el fichero con `pd.read_csv(RUTA_SUCIO)` dentro de un `try/except` y muestra el tipo de error y el mensaje.
2. Lee los bytes del fichero (`RUTA_SUCIO.read_bytes()`) e intenta decodificarlos con `'utf-8'`, `'latin-1'` y `'cp1252'`. ¿Con cuáles funciona?
3. Muestra las tres primeras líneas del fichero ya decodificado. ¿Qué separador usa?

> Opcional: la librería `charset_normalizer` (viene instalada con `requests`) puede adivinar el encoding: `from charset_normalizer import from_path; from_path(RUTA_SUCIO).best().encoding`.

In [ ]:
# TODO: intento de carga con try/except

# TODO: probar los tres encodings

# TODO: primeras líneas

**Parte B — Carga «en crudo».**
Para ver los datos **exactamente** como vienen, sin que pandas interprete nada, cárgalos así y guárdalos en `sucio_raw` (no la modifiques) y una copia de trabajo en `sucio`:

```python
pd.read_csv(RUTA_SUCIO, sep=..., encoding=..., dtype=str, keep_default_na=False)
```

Muestra las dimensiones, 10 filas al azar (`random_state=0`) e `info()`.

In [ ]:
# TODO: sucio_raw y sucio

**Parte C — ¿Qué no es un número?**
Escribe una función `no_numericos(serie)` que devuelva un `value_counts()` de los valores **no vacíos** de la serie que `pd.to_numeric(..., errors='coerce')` no consigue convertir. Aplícala a las columnas que deberían ser numéricas (`Age`, `RestingBP`, `Cholesterol`, `FastingBS`, `MaxHR`, `Oldpeak`, `HeartDisease`) y resume el resultado en una tabla con el número de valores no convertibles y hasta 5 ejemplos distintos.

In [ ]:
COLS_NUM_SUCIO = ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']

# TODO: función no_numericos y tabla resumen

**Preguntas:**

1. ¿Qué encoding tiene el fichero? ¿Por qué falla `utf-8` y, sin embargo, `latin-1` «funciona» siempre? (Pista: ¿puede fallar alguna vez `latin-1` al decodificar?) ¿Qué programa suele generar ficheros así?
2. ¿Por qué cargamos con `dtype=str` y `keep_default_na=False`? ¿Qué habría hecho pandas con los valores `'N/A'` y `''` si no hubiéramos puesto `keep_default_na=False`?
3. Según la Parte C, ¿qué tipos de problema hay en las columnas numéricas? Agrúpalos (símbolos o unidades, separador decimal, ausentes codificados, booleanos en texto, errores de tecleo…).
4. ¿Qué volumetría tiene el fichero? ¿Cuadra con los 918 pacientes de `heart.csv`? ¿Qué te hace sospechar?

*Escribe aquí tus respuestas:*

## Ejercicio 2.2 — Duplicados: exactos, reenviados y contradictorios

**Parte A — Duplicados exactos.**
Cuenta las filas que son duplicado **exacto** de otra y muéstralas junto a su original (`keep=False`), ordenadas por `id_paciente`.

In [ ]:
# TODO: duplicados exactos

**Parte B — Duplicados por identificador.**

1. Quita los duplicados exactos y guarda el resultado en `sucio_sin_exactos`.
2. Busca los `id_paciente` que siguen apareciendo más de una vez.
3. Para cada uno de esos identificadores, averigua **qué columnas** tienen valores distintos entre sus filas. Muestra una tabla `id_paciente → columnas que difieren`.

In [ ]:
# TODO: sucio_sin_exactos, ids repetidos y columnas que difieren

**Parte C — Decisión.**
Hay dos tipos de duplicado por identificador: los que solo cambian de **formato** (el centro reenvió el registro) y los que tienen un **dato clínico distinto**.

1. Crea en `sucio_sin_exactos` una columna booleana `revisar` que valga `True` en las filas de los identificadores con algún dato clínico distinto.
2. Crea `sucio_dedup` quedándote con la **primera** aparición de cada `id_paciente`.
3. Comprueba que `sucio_dedup` tiene una fila por paciente.

In [ ]:
# TODO: columna revisar, sucio_dedup y comprobación

**Preguntas:**

1. ¿Por qué `duplicated()` no detecta los registros reenviados? ¿Qué habría pasado si hubieras normalizado antes el texto y las fechas?
2. En los duplicados contradictorios, ¿cuál de los dos valores es el correcto? ¿Puedes saberlo con los datos que tienes? ¿Qué harías en un proyecto real?
3. Quedarse con la «primera aparición» depende del orden de las filas. ¿Qué criterio sería mejor si el fichero tuviera, por ejemplo, una fecha de modificación del registro?
4. En el Bloque 2 no podías saber si un paciente aparecía dos veces. ¿Qué ha cambiado aquí?

*Escribe aquí tus respuestas:*

## Ejercicio 2.3 — Texto y categorías: normalizar y validar

A partir de aquí trabaja sobre `sucio_dedup`. Crea `limpio = sucio_dedup.copy()` y ve corrigiendo sobre `limpio`.

**Parte A — Inventario.**
Muestra el `value_counts()` de `centro`, `Sex`, `ChestPainType`, `RestingECG`, `ST_Slope`, `FastingBS` y `ExerciseAngina`.

In [ ]:
COLS_TEXTO = ['centro', 'Sex', 'ChestPainType', 'RestingECG', 'ST_Slope', 'FastingBS', 'ExerciseAngina']

# TODO: limpio = ...  y value_counts de cada columna

**Parte B — Normalización.**

1. Escribe una función `normalizar_texto(serie)` que quite los espacios al principio y al final, convierta los espacios múltiples en uno solo, pase todo a minúsculas y convierta las cadenas vacías en `NaN`.
2. Aplícala a las columnas de `COLS_TEXTO` y vuelve a mirar sus valores distintos.
3. Escribe un diccionario de mapeo por columna que lleve cada variante a su valor **canónico** y aplícalo:
   - `centro` → `'Hospital Norte'`, `'Clínica Sur'`, `'Hospital Este'`
   - `Sex` → `'M'`, `'F'` · `ChestPainType` → `'TA'`, `'ATA'`, `'NAP'`, `'ASY'` · `RestingECG` → `'Normal'`, `'ST'`, `'LVH'` · `ST_Slope` → `'Up'`, `'Flat'`, `'Down'`
   - `ExerciseAngina` → `'Y'`, `'N'` · `FastingBS` → `0`, `1`

In [ ]:
# TODO: función normalizar_texto

# TODO: MAPEOS = {'centro': {...}, 'Sex': {...}, ...} y aplicación

**Parte C — Validación contra el dominio.**
Escribe una función `validar_categorias(df, dominios)` que, para cada columna de `dominios` (un diccionario `columna → lista de valores permitidos`), devuelva el número de valores **no nulos** que no están en la lista y algunos ejemplos. Aplícala a `limpio`: todas las columnas deben dar 0. Si alguna no da 0, vuelve a la Parte B.

In [ ]:
# TODO: DOMINIOS y validar_categorias

**Parte D — Tipos.**
Convierte `centro`, `Sex`, `ChestPainType`, `RestingECG` y `ExerciseAngina` a `category`, `ST_Slope` a una categórica **ordenada** (usa el orden que decidiste en el Bloque 2) y `FastingBS` a `Int64` (entero que admite ausentes). Muestra `limpio.dtypes`.

In [ ]:
# TODO: conversión de tipos

**Preguntas:**

1. ¿Por qué conviene normalizar (minúsculas, espacios) **antes** de escribir el diccionario de mapeo? ¿Cuántas entradas te habría hecho falta si no?
2. `'H. Norte'` o `'Hosp. Este'` no se arreglan normalizando. ¿Qué otros casos has tenido que resolver «a mano»? ¿Qué riesgo tiene automatizar estas correcciones (por ejemplo, con similitud de cadenas)?
3. ¿Por qué el mapeo es más seguro que algo como `str.title()` para dejar los valores canónicos? Piensa en `'Clínica Sur'` y `'Clinica Sur'`.
4. Cada centro codificaba `FastingBS` y `ExerciseAngina` a su manera. ¿Qué problema habría si un modelo recibiera `'Sí'`, `'SI'`, `'si'` y `1` como categorías distintas?

*Escribe aquí tus respuestas:*

## Ejercicio 2.4 — Números, unidades y fechas

**Parte A — Limpiar los números.**
Escribe una función `limpiar_numero(valor)` que convierta a `float` los valores de las columnas numéricas y devuelva `NaN` para los que no sean números. Tiene que resolver todos los casos que encontraste en el ejercicio 2.1 C.

> ⚠️ **Ojo con la función de la teoría** (Ejemplo 5): borra los puntos porque los considera separador de miles. ¿Sirve aquí? Hay centros que escriben `1.5` y otros `1,5`.

Aplícala a las columnas de `COLS_NUM_SUCIO` de `limpio` y vuelve a pasar `no_numericos` (adaptándolo si hace falta) para comprobar que no queda nada sin convertir.

In [ ]:
# TODO: función limpiar_numero y aplicación

**Parte B — Valores imposibles.**
Aplica `fuera_de_dominio(limpio, REGLAS)` del ejercicio 1.5 y muestra las filas afectadas de cada variable (solo las columnas `id_paciente`, `centro` y la variable).

In [ ]:
# TODO: fuera de dominio y filas afectadas

**Parte C — Corregir con trazabilidad.**
Cada corrección tiene que quedar **documentada**. Crea una lista `registro_cambios` y, cada vez que cambies un valor, añade un diccionario con `id_paciente`, `columna`, `valor_original`, `valor_nuevo` y `motivo`. Después:

1. **Colesterol en otra unidad.** Hay valores de colesterol pequeños con decimales. ¿En qué unidad están? Conviértelos a mg/dL (1 mmol/L = 38,67 mg/dL) y redondea a entero.
2. **Ceros.** Los ceros de `RestingBP` y `Cholesterol` pasan a `NaN`.
3. **Errores de tecleo.** Para el resto de valores fuera de dominio, decide en cada caso si la corrección es **inequívoca** (y la haces) o no (y pones `NaN`). Haz lo mismo con el valor de `RestingBP` que se perdió en la Parte A.
4. Muestra `registro_cambios` como DataFrame, cuenta los cambios por columna y motivo, y vuelve a aplicar `fuera_de_dominio`: no debe quedar nada.
5. Convierte a `Int64` las columnas que son enteras: `Age`, `RestingBP`, `Cholesterol`, `MaxHR` y `HeartDisease`.

In [ ]:
registro_cambios = []

def cambiar(df, mascara, columna, valor_nuevo, motivo):
    # Cambia los valores de `columna` en las filas de `mascara` y lo apunta en registro_cambios.
    # `valor_nuevo` puede ser un valor fijo o una función que recibe el valor original.
    for i in df.index[mascara]:
        original = df.at[i, columna]
        nuevo = valor_nuevo(original) if callable(valor_nuevo) else valor_nuevo
        registro_cambios.append({'id_paciente': df.at[i, 'id_paciente'], 'columna': columna,
                                 'valor_original': original, 'valor_nuevo': nuevo, 'motivo': motivo})
        df.at[i, columna] = nuevo

# TODO: correcciones usando cambiar(...)

**Parte D — Fechas.**

1. Mira algunos valores de `fecha_visita` de cada centro: ¿qué formato usa cada uno?
2. Conviértela a fecha con `pd.to_datetime(..., format='mixed', dayfirst=True, errors='coerce')` en una columna nueva, `fecha`.
3. Muestra los valores originales que han quedado como `NaT` y los que son **posteriores** a `FECHA_EXPORTACION`. Pon estos últimos a `NaT`, apuntándolo en `registro_cambios`.
4. Busca en `sucio_dedup` el valor `'2025-13-05'`. ¿Qué ha hecho pandas con él?
5. Muestra la fecha mínima y máxima resultante y elimina la columna `fecha_visita` (te quedas con `fecha`).

In [ ]:
# TODO: formatos por centro

# TODO: conversión, NaT, fechas futuras

# TODO: el caso '2025-13-05'

**Preguntas:**

1. ¿Qué habría pasado con `Oldpeak` si hubieras usado la función `limpiar_numero` de la teoría tal cual? ¿Por qué ese error es especialmente peligroso? (Pista: ¿daría algún aviso?)
2. ¿De qué centro vienen los colesteroles en mmol/L? Si no los hubieras convertido, ¿los habría detectado el criterio IQR? ¿Y la regla de dominio? ¿Habría alguna forma de detectar un colesterol en mmol/L «normal» si no tuviera decimales?
3. Explica qué correcciones de la Parte C consideras inequívocas y cuáles no. ¿Por qué no hemos corregido la edad 520 a 52?
4. ¿Qué ha hecho pandas con `'2025-13-05'`? ¿Por qué es peor un error silencioso que un `NaT`? ¿Cómo lo evitarías? (Pista: convertir cada centro con su `format=` explícito.)
5. ¿Para qué sirve `registro_cambios`? ¿Quién lo podría necesitar dentro de seis meses?

*Escribe aquí tus respuestas:*

## Ejercicio 2.5 — Evaluación de la cobertura

**Parte A — Cobertura por variable y global.**
Calcula la cobertura (% de valores presentes) de cada variable de `limpio` (sin contar `revisar`) y la cobertura global. Puedes reutilizar la función `informe_cobertura` del Ejemplo 11 de la teoría, con `umbral_variable=80` y `umbral_registro=60`.

> Recuerda que el colesterol que faltaba en `heart.csv` como 0, aquí venía como `0`, `'?'` o vacío: en `limpio` ya son todos `NaN`. Antes de empezar, comprueba que no queda ninguna columna con cadenas vacías sin convertir (mira `observaciones`).

In [ ]:
# TODO: cobertura por variable y global

**Parte B — Cobertura por centro.**
Calcula la cobertura de cada variable **en cada centro** y dibújala como un mapa de calor (centros en filas, variables en columnas, valores escritos). Muestra también cuántos pacientes ha enviado cada centro y su porcentaje de enfermos.

In [ ]:
# TODO: cobertura por centro, heatmap y pacientes por centro

**Parte C — Cobertura por registro.**

1. Calcula la cobertura de cada fila y dibuja su histograma con una línea en el umbral del 60 %.
2. Muestra las filas por debajo del umbral. ¿Qué tienen en común?
3. Repite el cálculo **sin** la columna `observaciones`. ¿Cambia mucho la distribución?

In [ ]:
# TODO: cobertura por registro

**Preguntas:**

1. ¿Qué variables quedan por debajo del 80 %? ¿Descartarías alguna? Ten en cuenta para qué sirve cada una.
2. `Cholesterol` está justo en el límite. Mirando la Parte B, ¿es un problema de una variable o de un **centro**? Si descartaras las filas sin colesterol, ¿qué le pasaría a la representación de cada centro en el dataset?
3. ¿Qué harías con los registros de la Parte C? Recuerda que conservan la variable objetivo.
4. **Completitud** y **cobertura** (en el sentido de «a quién representan los datos») no son lo mismo. Pon un ejemplo de este dataset en el que una variable esté completa al 100 % y, aun así, haya un problema de cobertura.

*Escribe aquí tus respuestas:*

## Ejercicio 2.6 — Comprobación final e informe de calidad

**Parte A — ¿Has limpiado bien?**
Los `id_paciente` corresponden, en orden, a las filas de `heart.csv`: `P0001` es la fila 0, `P0002` la fila 1, etc. Eso te permite **comprobar** tu limpieza contra el original (en un caso real no tendrías esta suerte).

1. Ordena `limpio` por `id_paciente` y alinea su índice con el de `heart_na` (sin la columna `chol_ausente`).
2. Para cada una de las 12 columnas de `heart.csv`, cuenta en cuántas filas tu valor **coincide** con el de `heart_na`, en cuántas **difiere** y en cuántas tú tienes un `NaN` que el original no tiene. Considera que dos `NaN` coinciden.
3. Muestra las filas en las que tu valor difiere del original.

In [ ]:
# TODO: comparación de limpio con heart_na

**Parte B — Exportar.**
Guarda `limpio` en `datos/heart_limpio.csv` (UTF-8, separador `,`, sin el índice) y `registro_cambios` en `datos/heart_registro_cambios.csv`. Vuelve a cargar `heart_limpio.csv` y comprueba que se lee bien con un `read_csv` sin parámetros. Aplica `informe_calidad(..., REGLAS)` del ejercicio 1.5 al fichero recargado.

In [ ]:
# TODO: exportar, recargar y aplicar informe_calidad

**Parte C — Informe de calidad.**
Completa el informe con **todos** los problemas que has encontrado en `heart_sucio.csv`, agrupados por dimensión de calidad:

| Dimensión | Problema | Variable(s) | Filas afectadas | Cómo lo detectaste | Decisión tomada |
|---|---|---|---|---|---|
| **Unicidad** (duplicados) | | | | | |
| **Validez** (formato y dominio) | | | | | |
| **Consistencia** (misma cosa, misma forma) | | | | | |
| **Exactitud** (¿el valor es el real?) | | | | | |
| **Completitud** (ausentes) | | | | | |
| **Codificación** (encoding, tipos) | | | | | |

Añade tantas filas como necesites.

**Preguntas:**

1. Según la Parte A, ¿en qué columnas te has quedado con valores distintos del original? ¿Se deben a errores tuyos o a decisiones razonables que no se podían tomar mejor?
2. ¿Qué problemas habrías sido **incapaz** de detectar sin tener el fichero original para comparar? ¿Qué te dice eso sobre los límites de la limpieza de datos?
3. ¿Qué le pedirías a cada centro para que la próxima exportación llegue sin estos problemas? Escribe al menos tres normas concretas.

*Escribe aquí tus respuestas:*